In [2]:
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
import math

In [2]:
TOPK = 50
sim_item_topk = 10
recall_item_num = 10

In [3]:
train_click = pd.read_csv('/kaggle/input/datasets/jiaweiwei0311/tianchinewsrec/train_click_log.csv')
testA_click = pd.read_csv('/kaggle/input/datasets/jiaweiwei0311/tianchinewsrec/testA_click_log.csv')
articles = pd.read_csv('/kaggle/input/datasets/jiaweiwei0311/tianchinewsrec/articles.csv').rename(columns={"article_id":"click_article_id"})
all_click = pd.concat([train_click,testA_click], ignore_index=True)
train_click.columns.tolist(),testA_click.columns.tolist(),articles.columns.tolist()

(['user_id',
  'click_article_id',
  'click_timestamp',
  'click_environment',
  'click_deviceGroup',
  'click_os',
  'click_country',
  'click_region',
  'click_referrer_type'],
 ['user_id',
  'click_article_id',
  'click_timestamp',
  'click_environment',
  'click_deviceGroup',
  'click_os',
  'click_country',
  'click_region',
  'click_referrer_type'],
 ['click_article_id', 'category_id', 'created_at_ts', 'words_count'])

In [6]:
TopK_articles = all_click['click_article_id'].value_counts().index[:TOPK]


In [9]:
item_sim = defaultdict(dict) # wij
item_click = defaultdict(int) # 物品被点击次数
user_click_items_dict = all_click.groupby('user_id')['click_article_id'].agg(list).to_dict()
for user,click_items in tqdm(user_click_items_dict.items()):
    for i in click_items:
        item_click[i]+=1
        for j in click_items:
            if i==j:
                continue
            item_sim[i].setdefault(j, 0)
            item_sim[i][j] += 1/math.log(1+len(click_items))



100%|██████████| 250000/250000 [00:23<00:00, 10834.32it/s]


In [12]:
for i, related in tqdm(item_sim.items()):
    for j in related:
        related[j]/=math.sqrt(item_click[i] * item_click[j])
# item_sim = {
#     i: {j: w / math.sqrt(item_click[i] * item_click[j]) for j, w in related.items()}
#     for i, related in item_sim.items()
# }


100%|██████████| 35198/35198 [00:04<00:00, 8696.65it/s] 


In [ ]:
topk_sim = {
    i:sorted(related.items(), key=lambda x:x[1], reverse= True)[:sim_item_topk]
    for i, related in item_sim.items()
}
user_recall_items_dict = {}
user_id_list=testA_click['user_id'].unique()
for user_id in tqdm(user_id_list): # 对每个用户召回
    item_rank = defaultdict(int)
    history=user_click_items_dict[user_id]
    history_set=set(history)
    for history_item_id in history:
        for j,sim in topk_sim.get(history_item_id,[]): 
            if j in history_set:
                continue
            item_rank[j]+=sim
    if len(item_rank)<sim_item_topk:
        for rank,j in enumerate(TopK_articles):
            if j in item_rank:
                continue
            item_rank[j]=-rank-1
            if len(item_rank)==sim_item_topk:
                break
    user_recall_items_dict[user_id]=sorted(item_rank.items(), key=lambda x:x[1], reverse=True)[:recall_item_num]


100%|██████████| 50000/50000 [00:03<00:00, 15796.05it/s]


In [ ]:
# ============================================================
# 环境 bootstrap：固定从 GitHub clone 项目
# 所有需要项目代码的单元格都依赖这一格，先运行它
# ============================================================
import os, sys, subprocess

REPO_URL = 'https://github.com/hydrofluoric07/news_recommender.git'
PROJ = '/kaggle/working/news_recommender'
UPDATE = False          # 改了 GitHub 上的代码后设成 True 再跑一次，拉最新版

if not os.path.exists(os.path.join(PROJ, 'main.py')):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, PROJ],
                   check=True, timeout=180,
                   env=dict(os.environ, GIT_TERMINAL_PROMPT='0'))
    print('已 clone 到', PROJ)
elif UPDATE:
    subprocess.run(['git', '-C', PROJ, 'pull', '--ff-only'],
                   check=True, timeout=180,
                   env=dict(os.environ, GIT_TERMINAL_PROMPT='0'))
    print('已更新', PROJ)

if PROJ not in sys.path:
    sys.path.insert(0, PROJ)
os.chdir('/kaggle/working')      # ./cache、./results 写到可写目录
print('项目目录:', PROJ)

In [ ]:
# ============================================================
# 线下评估：Recall@10 / MRR@5（ItemCF baseline）
# 严格口径：只用 train 点击；每人最后一条点击当答案，其余当历史
# 依赖上面的 bootstrap 单元格
# ============================================================
import os
import pandas as pd

from data_processing import get_all_click_df, resolve_data_path
from utils import get_user_item_time_dict, get_item_topk_click
from models.recall_baseline import itemcf_sim, generate_user_recall_dict

data_path = resolve_data_path()
cache_dir = os.environ.get('NEWS_CACHE_DIR', './cache/')

# ---- 1. 切分：每人最后一条当答案 ----
off = get_all_click_df(data_path=data_path, offline=True)
off = off.sort_values(['user_id', 'click_timestamp'])
val_df = off.groupby('user_id').tail(1)      # 答案
trn_df = off.drop(val_df.index)              # 历史
print(f'总点击 {len(off)} | 历史 {len(trn_df)} | 答案 {len(val_df)} | 用户 {val_df.user_id.nunique()}')

# ---- 2. 只用 trn 构建历史/热门/相似度（用 val 就是泄露）----
uit_tr = get_user_item_time_dict(trn_df)
hot_tr = get_item_topk_click(trn_df, k=50)
i2i_tr = itemcf_sim(trn_df, save_path=os.path.join(cache_dir, 'itemcf_sim_offline.pkl'),
                    use_cache=False)

# 只有 1 次点击的用户，切完就没有历史；补空列表，走热门补全
for u in val_df['user_id'].unique():
    uit_tr.setdefault(u, [])

# ---- 3. 对 val 用户生成召回 ----
val_users_df = pd.DataFrame({'user_id': val_df['user_id'].unique()})
recall_off = generate_user_recall_dict(
    val_df=val_users_df,
    user_item_time_dict=uit_tr,
    i2i_sim=i2i_tr,
    sim_item_topk=10,
    recall_item_num=10,
    item_topk_click=hot_tr,
    save_path=os.path.join(cache_dir, 'user_recall_items_offline.pkl'),
    use_cache=False,
)

# ---- 4. 指标 ----
gt = dict(zip(val_df['user_id'], val_df['click_article_id']))
n = len(recall_off)

def recall_at_k(k):
    hit = sum(1 for u, lst in recall_off.items() if gt.get(u) in [i for i, _ in lst[:k]])
    return hit / n

def mrr_at_k(k):
    total = 0.0
    for u, lst in recall_off.items():
        for rank, (item, _) in enumerate(lst[:k], 1):
            if item == gt.get(u):
                total += 1.0 / rank
                break
    return total / n

print(f'评估用户数: {n}')
print(f'Recall@10 = {recall_at_k(10):.4f}')
print(f'MRR@5     = {mrr_at_k(5):.4f}')

In [19]:
from collections import Counter
gt = dict(zip(val_df.user_id, val_df.click_article_id))

ranks = []
for u, lst in recall_off.items():
    ids = [i for i, _ in lst[:10]]
    ranks.append(ids.index(gt[u]) + 1 if gt[u] in ids else 0)

print('名次分布(0=未进前10):', dict(sorted(Counter(ranks).items())))
print('Recall@5 =', sum(1 for r in ranks if 1 <= r <= 5) / len(ranks))
print('MRR@10   =', sum(1 / r for r in ranks if r) / len(ranks))

hot10 = list(hot_tr[:10])
print('只推热门Top10 的 Recall@10 =', sum(1 for u in recall_off if gt.get(u) in hot10) / len(recall_off))
print('冷启动用户数(无历史) =', sum(1 for u in recall_off if not uit_tr.get(u)))


NameError: name 'recall_off' is not defined

In [ ]:
# ============================================================
# 对照实验：无加权 ItemCF vs 加权 ItemCF
# 同一份切分、同一批用户、同一个指标函数，只换模型
# 依赖上面的 bootstrap 单元格；相似度复用前面算好的缓存
# ============================================================
import os, time
import pandas as pd

from data_processing import get_all_click_df, get_item_info_df, resolve_data_path
from utils import get_user_item_time_dict, get_item_topk_click, get_item_info_dict
from models.recall_baseline import itemcf_sim as itemcf_sim_base, generate_user_recall_dict
from models.itemcf_recall import itemcf_sim as itemcf_sim_w, generate_itemcf_recall_dict

data_path = resolve_data_path()
cache_dir = os.environ.get('NEWS_CACHE_DIR', './cache/')

# ---- 1. 切分（和线下评估完全一致）----
off = get_all_click_df(data_path=data_path, offline=True) \
        .sort_values(['user_id', 'click_timestamp'])
val_df = off.groupby('user_id').tail(1)
trn_df = off.drop(val_df.index)
gt = dict(zip(val_df['user_id'], val_df['click_article_id']))

uit_tr = get_user_item_time_dict(trn_df)
hot_tr = get_item_topk_click(trn_df, k=50)
_, _, item_created_time_dict = get_item_info_dict(get_item_info_df(data_path))

# ---- 2. 相似度：复用线下评估 / 加权单元格已经算好的缓存 ----
t0 = time.time()
i2i_base = itemcf_sim_base(trn_df,
                           save_path=os.path.join(cache_dir, 'itemcf_sim_offline.pkl'),
                           use_cache=True)
i2i_w = itemcf_sim_w(trn_df, item_created_time_dict,
                     os.path.join(cache_dir, 'itemcf_w_offline.pkl'),
                     use_cache=True)
print(f'相似度加载/计算完成，用时 {time.time() - t0:.0f}s')

def top5(d):
    return [i for i, _ in sorted(d.items(), key=lambda x: -x[1])[:5]]

sample_items = list(i2i_base)[:500]
changed = sum(top5(i2i_base[i]) != top5(i2i_w[i]) for i in sample_items)
print(f'抽样 {len(sample_items)} 个物品，加权后 top-5 邻居变化: {changed}')

# ---- 3. 同一批用户（先 1 万；跑通后可以调大）----
N_USERS = 10000
users_eval = pd.DataFrame({'user_id': val_df['user_id'].unique()[:N_USERS]})
for u in users_eval['user_id']:
    uit_tr.setdefault(u, [])
print(f'评估用户 {len(users_eval)}')

# ---- 4. 两个模型，用同一份历史 ----
t0 = time.time()
recall_base = generate_user_recall_dict(
    val_df=users_eval, user_item_time_dict=uit_tr, i2i_sim=i2i_base,
    sim_item_topk=10, recall_item_num=10, item_topk_click=hot_tr,
    save_path=os.path.join(cache_dir, 'recall_base_cmp.pkl'), use_cache=False)
print(f'baseline 召回用时 {time.time() - t0:.0f}s')

t0 = time.time()
recall_w = generate_itemcf_recall_dict(
    val_df=users_eval, user_item_time_dict=uit_tr, i2i_sim=i2i_w,
    sim_item_topk=10, recall_item_num=10, item_topk_click=hot_tr,
    item_created_time_dict=item_created_time_dict,
    emb_i2i_sim={},                      # 先关掉内容相似度，只比加权本身
    save_path=os.path.join(cache_dir, 'recall_w_cmp.pkl'), use_cache=False)
print(f'weighted 召回用时 {time.time() - t0:.0f}s')

# ---- 5. 同一套指标 ----
def eval_recall(recall, gt, k_recall=10, k_mrr=5):
    ranks = []
    for u, lst in recall.items():
        ids = [i for i, _ in lst[:k_recall]]
        ranks.append(ids.index(gt[u]) + 1 if u in gt and gt[u] in ids else 0)
    n = len(ranks)
    return (sum(1 for r in ranks if 1 <= r <= k_recall) / n,
            sum(1 / r for r in ranks if 1 <= r <= k_mrr) / n)

print('\n模型        Recall@10   MRR@5')
for name, rec in [('baseline', recall_base), ('weighted', recall_w)]:
    r10, m5 = eval_recall(rec, gt)
    print(f'{name:10s}  {r10:.4f}      {m5:.4f}')

same = sum([i for i, _ in recall_base[u][:5]] == [i for i, _ in recall_w[u][:5]]
           for u in recall_base)
print(f'\ntop-5 完全相同的用户: {same}/{len(recall_base)} ({same / len(recall_base):.2%})')